In [1]:
# imports
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# set which one-hot sample to compare (0-4)
# 0: corner (0,0)
# 1: corner (47,47)
# 2: center (24,24)
# 3: top edge (0,24)
# 4: left edge (24,0)
TEST_NUM = 1

In [ ]:
def load_rtl_layer(path, nfrac):
    raw = np.loadtxt(path, delimiter=',')   # shape (n_positions, n_channels)
    flat = raw.flatten()                     # row-major: channel varies fastest, matches (H,W,C) flatten
    return flat / (2 ** nfrac)

def load_ref_layer(path):
    return np.loadtxt(path, delimiter=',')   # already flat, already decimal

# every layer in pipeline order:
#   (short name, rtl filename suffix, hls4ml filename suffix, nfrac)
LAYER_SPECS = [
    ("conv0",  "conv0",  "q_activation",    6),
    ("pool0",  "pool0",  "max_pooling2d",   6),
    ("conv1",  "conv1",  "q_activation_1",  6),
    ("pool1",  "pool1",  "max_pooling2d_1", 6),
    ("conv2",  "conv2",  "q_activation_2",  6),
    ("pool2",  "pool2",  "max_pooling2d_2", 6),
    ("dense0", "dense0", "q_dense",         5),
    ("relu0",  "relu0",  "q_activation_3",  5),
    ("dense1", "dense1", "q_dense_1",       5),
    ("relu1",  "relu1",  "q_activation_4",  5),
    ("final",  "final",  "q_dense_2",       5),
]

def rtl_path(test_num, suffix):
    return f"traces/rtl_{test_num}_{suffix}.csv"

def hls_path(test_num, suffix):
    return f"traces/hls4ml_{test_num}_{suffix}.csv"

def keras_path(test_num, suffix):
    return f"traces/keras_{test_num}_{suffix}.csv"

In [4]:
# quick single-layer sanity check (conv0) for the selected TEST_NUM
rtl_conv0 = load_rtl_layer(rtl_path(TEST_NUM, "conv0"), nfrac=6)
hls4ml_conv0 = load_ref_layer(hls_path(TEST_NUM, "q_activation"))

print(rtl_conv0.shape, hls4ml_conv0.shape)    # RTL may be a few positions short
                                              # if finalOutputValid fired (and the testbench
                                              # closed the trace files) before conv0 finished
                                              # streaming its full raster scan -- expected,
                                              # not a bug.

n = min(len(rtl_conv0), len(hls4ml_conv0))
diff = np.abs(rtl_conv0[:n] - hls4ml_conv0[:n])
print("mean:", diff.mean())
print("median:", np.median(diff))
print("p95:", np.percentile(diff, 95))
print("max:", diff.max())
print(f"num outliers (>0.1): {(diff > 0.1).sum()} / {diff.size}")

(12684,) (12696,)
mean: 0.0
median: 0.0
p95: 0.0
max: 0.0
num outliers (>0.1): 0 / 12684


In [5]:
def diff_stats(rtl_p, ref_p, nfrac, label=None):
    """
    Computes RTL vs. reference (hls4ml or keras) error stats for one layer.
    Trims to the shorter length if the RTL trace was cut a few positions short
    (expected when finalOutputValid fires before conv0 finishes its full raster
    scan -- the trailing positions are unused by the real output anyway).
    Returns a dict of stats so results can be collected across layers/tests.
    """
    rtl = load_rtl_layer(rtl_p, nfrac)
    ref = load_ref_layer(ref_p)
    n = min(len(rtl), len(ref))
    note = ""
    if len(rtl) != len(ref):
        note = f"  [trimmed: rtl={len(rtl)}, ref={len(ref)} -> {n}]"
    diff = np.abs(rtl[:n] - ref[:n])
    stats = {
        "label": label or ref_p,
        "mean": diff.mean(),
        "median": np.median(diff),
        "p95": np.percentile(diff, 95),
        "max": diff.max(),
        "outliers": int((diff > 0.1).sum()),
        "n": diff.size,
    }
    print(f"{stats['label']:12s} mean={stats['mean']:.6f} median={stats['median']:.6f} "
          f"p95={stats['p95']:.6f} max={stats['max']:.6f} outliers={stats['outliers']}/{stats['n']}{note}")
    return stats

def run_full_comparison(test_num, ref_source="hls4ml"):
    """
    Runs diff_stats for every layer in LAYER_SPECS, for the given TEST_NUM.
    ref_source: 'hls4ml' or 'keras' -- which reference trace to compare RTL against.
    Returns a list of per-layer stat dicts (in pipeline order).
    """
    print(f"=== TEST_NUM={test_num}  (ref={ref_source}) ===")
    results = []
    for name, rtl_suffix, hls_suffix, nfrac in LAYER_SPECS:
        rp = rtl_path(test_num, rtl_suffix)
        refp = hls_path(test_num, hls_suffix) if ref_source == "hls4ml" else keras_path(test_num, hls_suffix)
        results.append(diff_stats(rp, refp, nfrac, label=name))
    return results

results = run_full_comparison(TEST_NUM, ref_source="hls4ml")

=== TEST_NUM=1  (ref=hls4ml) ===
conv0        mean=0.000000 median=0.000000 p95=0.000000 max=0.000000 outliers=0/12684  [trimmed: rtl=12684, ref=12696 -> 12684]
pool0        mean=0.000000 median=0.000000 p95=0.000000 max=0.000000 outliers=0/726
conv1        mean=0.007812 median=0.000000 p95=0.062500 max=0.062500 outliers=0/648
pool1        mean=0.007812 median=0.000000 p95=0.062500 max=0.062500 outliers=0/128
conv2        mean=0.029687 median=0.000000 p95=0.296875 max=0.296875 outliers=4/40
pool2        mean=0.029687 median=0.000000 p95=0.163281 max=0.296875 outliers=1/10
dense0       mean=0.010417 median=0.000000 p95=0.031250 max=0.031250 outliers=0/15
relu0        mean=0.002083 median=0.000000 p95=0.009375 max=0.031250 outliers=0/15
dense1       mean=0.021875 median=0.031250 p95=0.048437 max=0.062500 outliers=0/10
relu1        mean=0.003125 median=0.000000 p95=0.017187 max=0.031250 outliers=0/10
final        mean=0.012500 median=0.000000 p95=0.031250 max=0.031250 outliers=0/5


In [ ]:
# summarize just the final-layer error per test.
def summarize_final_across_tests(test_nums=(0, 1, 2, 3, 4), ref_source="hls4ml"):
    print(f"{'test':>4s}  {'mean':>10s}  {'median':>10s}  {'p95':>10s}  {'max':>10s}  outliers")
    for t in test_nums:
        try:
            s = diff_stats(
                rtl_path(t, "final"),
                hls_path(t, "q_dense_2") if ref_source == "hls4ml" else keras_path(t, "q_dense_2"),
                nfrac=5,
                label=f"test {t}",
            )
        except OSError as e:
            print(f"test {t}: no data yet ({e})")

summarize_final_across_tests()

test        mean      median         p95         max  outliers
test 0       mean=0.012500 median=0.000000 p95=0.031250 max=0.031250 outliers=0/5
test 1       mean=0.012500 median=0.000000 p95=0.031250 max=0.031250 outliers=0/5
test 2       mean=0.068750 median=0.031250 p95=0.125000 max=0.125000 outliers=2/5
test 3       mean=0.012500 median=0.000000 p95=0.031250 max=0.031250 outliers=0/5
test 4       mean=0.050000 median=0.031250 p95=0.093750 max=0.093750 outliers=0/5
